# Printed-Range Trap: model runs

Runs one model on one prompt set. All the logic is in `src/` of the repository.

Kaggle settings: GPU T4 x2, internet on, `HF_TOKEN` secret attached (needed for MedGemma).

## Settings

In [ ]:
MODEL = "qwen"  # "qwen" or "medgemma"
HARNESS = "plain"  # "plain", "instruction" or "guidelines"

## Setup

In [ ]:
!pip install -q uv
!uv pip install --system -q "transformers==5.18.0" "accelerate==1.15.0"

In [ ]:
import os
import shutil

import torch
import transformers
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), "No GPU: set the accelerator to GPU T4 x2"
assert transformers.__version__ == "5.18.0", "Old transformers still loaded: restart the session"

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:  # noqa: BLE001 - no secret attached; fine for Qwen
    print("No HF_TOKEN secret (fine for Qwen)")

REPO_DIR = "/kaggle/working/printed-range-trap"
shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone -q https://github.com/REZZOUGAICHA/printed-range-trap.git {REPO_DIR}
os.chdir(REPO_DIR)
!git log -1 --format="code version: %h %s"

## Prompts

In [ ]:
!python src/build_prompts.py --languages en fr --harness {HARNESS}

PROMPTS = "data/prompts.jsonl" if HARNESS == "plain" else f"data/prompts_{HARNESS}.jsonl"
!git diff --quiet {PROMPTS} && echo "prompts match the repository" || echo "WARNING: prompts differ from the repository"

## Model

In [ ]:
MODELS = {
    "qwen": {
        "name": "qwen3.5-4b",
        "args": "--model-id Qwen/Qwen3.5-4B --revision 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a "
        "--dtype float16 --device-map cuda:0",
    },
    "medgemma": {
        # float16 overflows on T4 (NaN logits), and its thinking token is suppressed
        "name": "medgemma-1.5-4b",
        "args": "--model-id google/medgemma-1.5-4b-it --revision 91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b "
        "--dtype float32 --device-map auto --suppress-tokens '<unused94>'",
    },
}
name, ARGS = MODELS[MODEL]["name"], MODELS[MODEL]["args"]

# Saved outside the code folder, so re-running the setup cell never deletes answers
RESULTS = "/kaggle/working/results"
if HARNESS == "plain":
    OUT = f"{RESULTS}/{name}/generations_{name}.jsonl"
else:
    OUT = f"{RESULTS}/{name}/{HARNESS}/generations_{name}_{HARNESS}.jsonl"
print(OUT)

## Run
A quick test on 2 prompts, then the full set. The run is resumable: if the session stops, run all cells again.

In [ ]:
!rm -f /tmp/test_run.jsonl
!python src/run_model.py {ARGS} --prompts {PROMPTS} --out /tmp/test_run.jsonl --limit 2

In [ ]:
!python src/run_model.py {ARGS} --prompts {PROMPTS} --out {OUT}

## Results

In [ ]:
!python src/metrics.py {OUT} | grep -E "deference|format_failure"

## Download
Unzip into the root of the local repository: the files land in `results/`.

In [ ]:
from IPython.display import FileLink

os.chdir("/kaggle/working")
!rm -f results.zip && zip -r -q results.zip results
FileLink("results.zip")